In [2]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.ssl_ import create_urllib3_context
import pandas as pd
import time
import sqlite3
from update_db_func import *
from sqlalchemy import create_engine

In [14]:
now=pd.Timestamp.now().replace(day=1).strftime('%Y-%m-%d')

In [ ]:
date_range = pd.date_range(end=now, periods=4, freq='ME')
date_range_str=[int(i.strftime('%Y%m')) for i in date_range]

[202512, 202601, 202602, 202603]

In [16]:
class TLSAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        # 手動創建 SSL Context 並指定支援的加密套件
        context = create_urllib3_context()
        # 允許較舊的加密算法 (下放安全性)
        context.set_ciphers('DEFAULT@SECLEVEL=1') 
        kwargs['ssl_context'] = context
        return super(TLSAdapter, self).init_poolmanager(*args, **kwargs)

def get_china_trade_by_country(date):
    s = requests.Session()
    s.mount('https://', TLSAdapter())
    url=r'https://data.mofcom.gov.cn/datamofcom/front/totalbycountry/query'
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36 Edg/129.0.0.0',
    }

    playload={'date': date}

    # 之後如常使用 s 進行請求
    response = s.post(url, headers=headers, data=playload)
    time.sleep(5)
    if response.status_code==200:
        data=response.json()['rows']
        df=pd.DataFrame(data)
        return df
    else:
        return pd.DataFrame()

In [17]:
dfs=[get_china_trade_by_country(i) for i in date_range_str]

In [18]:
df=pd.concat(dfs)
df['updated_at'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')

In [20]:
db_file_name=r'china_mocom.db'
db_table_name = r'by_country'
unique_keys = ['trade_date', 'type']
engine = create_engine(f'sqlite:///{db_file_name}')

# create_db(engine, db_table_name, df, unique_keys)
create_update_db(engine, db_table_name, df, unique_keys)

Table 'by_country' exists.
✅ Successfully synced 216 rows (handled special characters).


In [21]:
conn=sqlite3.connect(db_file_name)
df=pd.read_sql_query(f'SELECT * FROM {db_table_name}', conn)
conn.close()
df

,imexgap_lj_value,total_value,imexgap_per,type,trade_date,total_lj_per,import_lj_value,export_per,import_lj_per,import_per,imexgap_value,total_lj_value,export_lj_value,export_lj_per,id,import_value,imexgap_lj_per,total_per,export_value,updated_at
0,None,None,None,印度,202401,16.9,20.13,None,56.7,None,None,134.85,114.73,11.9,ff8080818e470813018e59d5eac0001c,None,None,None,None,2026-04-21 14:35:23
1,None,None,None,日本,202401,4.5,118.11,None,10.0,None,None,261.03,142.92,0.4,ff8080818e470813018e59d5eada0021,None,None,None,None,2026-04-21 14:35:23
2,None,None,None,韩国,202401,11.3,145.04,None,22.6,None,None,266.25,121.21,0.1,ff8080818e470813018e59d5ebb80032,None,None,None,None,2026-04-21 14:35:23
3,None,None,None,俄罗斯,202401,12.5,104.65,None,10.4,None,None,196.88,92.23,15.0,ff8080818e470813018e59d5efbd00aa,None,None,None,None,2026-04-21 14:35:23
4,None,None,None,南非,202401,27.3,39.76,None,77.1,None,None,57.94,18.18,-21.2,ff8080818e470813018e59d5ee2a0072,None,None,None,None,2026-04-21 14:35:23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
463,,,,意大利,202602,27.9,39.57,,10.6,,,138.49,98.93,36.4,53d8a9ef9d232675019d42a27f43019e,,,,,2026-04-21 14:41:18
464,,,,沙特阿拉伯,202602,5.5,78.77,,-7.3,,,176.84,98.07,18.7,53d8a9ef9d232675019d42a27c110140,,,,,2026-04-21 14:41:18
465,,,,土耳其,202602,17.6,8.35,,13.7,,,84.10,75.75,18.1,53d8a9ef9d232675019d42a27c440146,,,,,2026-04-21 14:41:18
466,,,,阿根廷,202602,60.5,26.09,,206.0,,,47.81,21.72,2.1,53d8a9ef9d232675019d42a2811701ce,,,,,2026-04-21 14:41:18


In [ ]:
import plotly.express as px

df['date']=pd.to_datetime(df['trade_date'], format='%Y%m')
# df=df.sort_values('date', ascending=False)
df['import_lj_value']=pd.to_numeric(df['import_lj_value'])
df['export_lj_value']=pd.to_numeric(df['export_lj_value'])
df = df.sort_values(['type', 'date'])
df['year'] = df['date'].dt.year
df['import_monthly'] = df.groupby(['year', 'type'])['import_lj_value'].diff()
df['export_monthly'] = df.groupby(['year', 'type'])['export_lj_value'].diff()
df['import_monthly'] = df['import_monthly'].fillna(df['import_lj_value'])
df['export_monthly'] = df['export_monthly'].fillna(df['export_lj_value'])
px.bar(df, x='date', y='import_monthly', color='type', title='中國入口情況')


In [55]:
px.bar(df, x='date', y='export_monthly', color='type', title='中國出口情況')

In [57]:
df['import_monthly'] = df['import_monthly'].fillna(df['import_lj_value'])
df['export_monthly'] = df['export_monthly'].fillna(df['export_lj_value'])
df['net_export_import_monthly']=df['export_monthly']-df['import_monthly']
px.bar(df, x='date', y='net_export_import_monthly', color='type', title='中國淨入口情況')